In [2]:
print("hello")

hello


In [3]:
%pip install youtube-transcript-api faiss-cpu

  Using cached youtube_transcript_api-1.2.4-py3-none-any.whl.metadata (24 kB)
  Using cached faiss_cpu-1.15.0-cp314-cp314-win_amd64.whl.metadata (7.8 kB)
Using cached youtube_transcript_api-1.2.4-py3-none-any.whl (485 kB)
Using cached faiss_cpu-1.15.0-cp314-cp314-win_amd64.whl (16.5 MB)

   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   ---------------------------------------- 0/2 [faiss-cpu]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   ---------------------------------------- 2/2 [youtube-transcript-api]


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sys
print(sys.executable)

d:\Gen_AI\LangChain_Models\venv\Scripts\python.exe


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter          # not langchain.text_splitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\GEO COMPUTER S\AppData\Local\Temp\ipykernel_10196\559133239.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - Indexing (Document Ingestion)

In [9]:
with open('transcript.txt', 'r', encoding='utf-8') as f:
    transcript = f.read()

print(f"Loaded transcript: {len(transcript.split())} words, {len(transcript)} characters")

Loaded transcript: 19927 words, 129951 characters


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# TRANSCRIPT SOURCE — why we are NOT using the YouTube API here
# ─────────────────────────────────────────────────────────────────────────────
# The ORIGINAL plan was to pull the transcript live from YouTube using the
# youtube-transcript-api library. That code is CORRECT — but it fails at the
# NETWORK level, not the code level:
#
#   - On Colab: YouTube blocks cloud-provider IPs outright -> RequestBlocked
#   - Locally (this machine): YouTube ALSO blocked this IP -> IpBlocked
#
# Both are YouTube refusing the connection based on WHERE the request came
# from, not anything wrong with the request itself. Fixing this properly would
# require a paid residential proxy (the library's own recommended workaround),
# which is out of scope for a learning exercise.
#
# --- ORIGINAL YOUTUBE VERSION (kept for reference — NOT executed) ---
# video_id = "Gfr50f6ZBvo"
# try:
#     ytt_api = YouTubeTranscriptApi()
#     fetched = ytt_api.fetch(video_id, languages=["en"])
#     transcript = " ".join(snippet.text for snippet in fetched)
#     print(transcript)
# except TranscriptsDisabled:
#     print("No captions available for this video.")
#
# RESULT WHEN RUN: RequestBlocked / IpBlocked — connection reaches YouTube,
# YouTube refuses it based on IP reputation.
#
# WHAT WE DID INSTEAD:
# RAG does not care WHERE its source text comes from — YouTube is just one
# possible document source. So we generated a long, hours-long-video-scale
# transcript (~20k words / ~130k characters) covering the same subject matter
# (AI history, DeepMind, RAG mechanics) and load it from transcript.txt
# instead. Every step downstream — splitting, embedding, storing, retrieving,
# generating — works identically regardless of the ingestion source. This is
# the SAME "documented but not executed" pattern used earlier in the repo for
# the local HuggingFacePipeline reference code.
# ─────────────────────────────────────────────────────────────────────────────

print("Using local transcript.txt as the document source (see comments above for why).")

Using local transcript.txt as the document source (see comments above for why).


In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [15]:
len(chunks)

158

In [16]:
chunks[100]

Document(metadata={}, page_content="[slide transition]\n\nHere's another angle on the same idea. To be clear about scope, this session does not cover nuclear fusion. That is a topic in energy and physics, interesting in its own right, but entirely outside what we are discussing today, which is artificial intelligence, DeepMind, and retrieval augmented generation. This is mentioned only because the question comes up occasionally.\n\n[slide transition]\n\nCircling back to an earlier point, Positional information also matters in transformers. Because attention looks at all tokens at once rather than in strict order, the model needs some signal about where each token sits in the sequence. Positional encodings, whether fixed sinusoidal patterns or learned embeddings, are added to each token so the model can distinguish the first word of a sentence from the last.\n\n[slide transition]")

## Step 1c & 1d - Indexing (Embedding Generation and Storing)

In [18]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=r"D:\Gen_AI\LangChain_Models\.env")

True

In [20]:
# embed every chunk and build the FAISS vector store in one call (your vector-store topic)
embeddings = HuggingFaceEndpointEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

In [21]:
vector_store.index_to_docstore_id   # internal mapping of vector index -> document id

{0: '17afd1ff-eb46-4c73-8c4a-9bdc69728015',
 1: 'b4806cf5-3bb1-4337-b746-be27ad3adf43',
 2: '0a199a93-c081-4b20-8261-92a01fe323da',
 3: '7fcb7777-b4b1-4e1f-be41-94a5c8b93744',
 4: '316f63af-0705-4e6f-be82-f5fce4005b8d',
 5: '15689043-cfc6-44ad-af87-c77173906554',
 6: '89a77c50-4ab8-47f5-9a48-4153a8d6fdb0',
 7: 'bb6336d7-769b-42e9-97d8-febd375c2ef3',
 8: 'dafd2521-894f-454a-97ae-0f04e9eb8e59',
 9: 'd0962be7-bf10-41e1-8a3e-75a1755c369a',
 10: '4f3f2620-55a0-4446-b425-f9ee7a6963d4',
 11: 'e72ef02f-19f2-4d4e-a3c6-d142211040b5',
 12: '5a202916-5029-4140-aa86-ba4e896cbdfb',
 13: 'cb4237cd-b7f3-4597-af3d-2dcf6d62d5cf',
 14: '15893a80-bf11-433c-b24c-b4a40c7bde99',
 15: '74b1ae96-4e08-4a7d-bbb3-d0e9ea442a18',
 16: '1463664c-f13a-44b6-822b-80596038606d',
 17: '3483d501-6e28-45de-b6b5-e2c979fe9555',
 18: '8165afff-1854-400b-a45b-cf0a4d150239',
 19: 'dc5baa8f-2ac4-44ca-8d2a-e7c224f2ac9d',
 20: 'e4791f8f-be05-4b25-94d4-5f453df13ef4',
 21: '81addfa4-cfe3-4a3f-a9e2-f74109226f0b',
 22: '9dacad51-ffec-

In [23]:
vector_store.get_by_ids(['7fcb7777-b4b1-4e1f-be41-94a5c8b93744'])
# NOTE: this id is from the original run. YOUR ids differ — grab one from cell 13's output.

[Document(id='7fcb7777-b4b1-4e1f-be41-94a5c8b93744', metadata={}, page_content='[brief technical difficulty, resuming]\n\nDeepMind is known for breakthroughs like AlphaGo, which defeated the world champion at the board game Go. Go has more possible positions than atoms in the observable universe, and experts thought a machine beating a top human was a decade away. In twenty sixteen, AlphaGo defeated Lee Sedol in a five game match watched by millions. Move thirty seven in the second game was so unexpected that commentators initially thought it was a mistake.\n\n[pause for questions]\n\nAfter AlphaGo came AlphaZero, which taught itself Go, chess, and shogi from scratch with no human games, purely through self play. Then came AlphaFold, which predicts the three dimensional structure of proteins from their amino acid sequence, cracking a fifty year old open problem in biology. DeepMind released predicted structures for nearly every known protein, free, to the scientific community.')]

## Step 2 - Retrieval

In [24]:
# wrap the store as a retriever (your retrievers topic): return the 4 nearest chunks
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [25]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEndpointEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002860568E270>, search_kwargs={'k': 4})

In [26]:
retriever.invoke('What is deepmind')   # -> 4 transcript chunks most relevant to the query

[Document(id='8c543fd7-cb81-4fcc-88b5-209dde83254d', metadata={}, page_content='[brief technical difficulty, resuming]\n\nCircling back to an earlier point, DeepMind is a British AI research lab founded in twenty ten and acquired by Google in twenty fourteen. It was founded by Demis Hassabis, Shane Legg, and Mustafa Suleyman. Demis Hassabis is the co founder and chief executive officer of DeepMind. He was a child chess prodigy, then a video game designer, then a neuroscientist, before turning to artificial intelligence, with the stated mission to solve intelligence and use it to solve everything else.\n\n[brief technical difficulty, resuming]'),
 Document(id='a119859d-b176-4d05-b35d-8b4cc4e82d53', metadata={}, page_content='[brief technical difficulty, resuming]\n\nDeepMind is a British AI research lab founded in twenty ten and acquired by Google in twenty fourteen. It was founded by Demis Hassabis, Shane Legg, and Mustafa Suleyman. Demis Hassabis is the co founder and chief executive 

## Step 3 - Augmentation

In [27]:
# temperature isn't a top-level arg on HuggingFaceEndpoint — pass it here
llm_endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    provider="cerebras",
    task="text-generation",
    temperature=0.2
)
llm = ChatHuggingFace(llm=llm_endpoint)

In [28]:
# the key RAG instruction: answer ONLY from retrieved context, else say "I don't know"
# {context} = retrieved chunks, {question} = user's question
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=['context', 'question']
)

In [29]:
question = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)   # fetch relevant chunks for THIS question

In [30]:
retrieved_docs

[Document(id='228dd061-41a2-452f-9650-04f291ff8dd5', metadata={}, page_content='[brief technical difficulty, resuming]\n\nTo be clear about scope, this session does not cover nuclear fusion. That is a topic in energy and physics, interesting in its own right, but entirely outside what we are discussing today, which is artificial intelligence, DeepMind, and retrieval augmented generation. This is mentioned only because the question comes up occasionally.\n\nGeneration is the final step: the augmented prompt, containing both retrieved context and the question, goes to the language model, which produces an answer grounded in the retrieved material rather than its own memory, making the answer more accurate and able to reflect information the model never saw during training.\n\n[slide transition]'),
 Document(id='ac0af4d5-835d-4443-8ad7-139a8d993ae4', metadata={}, page_content="[pause for questions]\n\nAs I mentioned earlier, but it's worth restating: As models and retrieval systems keep i

In [31]:
# join the retrieved chunks into a single block to drop into {context}
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"[brief technical difficulty, resuming]\n\nTo be clear about scope, this session does not cover nuclear fusion. That is a topic in energy and physics, interesting in its own right, but entirely outside what we are discussing today, which is artificial intelligence, DeepMind, and retrieval augmented generation. This is mentioned only because the question comes up occasionally.\n\nGeneration is the final step: the augmented prompt, containing both retrieved context and the question, goes to the language model, which produces an answer grounded in the retrieved material rather than its own memory, making the answer more accurate and able to reflect information the model never saw during training.\n\n[slide transition]\n\n[pause for questions]\n\nAs I mentioned earlier, but it's worth restating: As models and retrieval systems keep improving, the boundary between what a model knows innately and what it needs to look up will keep shifting, but the core idea will likely remain: separate the 

In [32]:
# inject context + question into the template -> a ready-to-send prompt
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [33]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      [brief technical difficulty, resuming]\n\nTo be clear about scope, this session does not cover nuclear fusion. That is a topic in energy and physics, interesting in its own right, but entirely outside what we are discussing today, which is artificial intelligence, DeepMind, and retrieval augmented generation. This is mentioned only because the question comes up occasionally.\n\nGeneration is the final step: the augmented prompt, containing both retrieved context and the question, goes to the language model, which produces an answer grounded in the retrieved material rather than its own memory, making the answer more accurate and able to reflect information the model never saw during training.\n\n[slide transition]\n\n[pause for questions]\n\nAs I mentioned earlier, but it's worth restating: As mode

## Step 4 - Generation

In [34]:
answer = llm.invoke(final_prompt)   # LLM answers using ONLY the injected context
print(answer.content)

No. The transcript repeatedly states that nuclear fusion is **not** covered in the session—​it is mentioned only to clarify that it lies outside the scope of the discussion on artificial intelligence, DeepMind, and retrieval‑augmented generation.


## Building a Chain

In [35]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [36]:
# same join as cell 24, but as a reusable function so it can go inside a chain
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [37]:
# runs both branches on the SAME input question:
#   'context'  -> retrieve chunks, then format them into a string
#   'question' -> pass the question straight through unchanged
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [38]:
parallel_chain.invoke('who is Demis')   # -> {'context': <chunks>, 'question': 'who is Demis'}

{'context': "[brief technical difficulty, resuming]\n\nDeepMind is a British AI research lab founded in twenty ten and acquired by Google in twenty fourteen. It was founded by Demis Hassabis, Shane Legg, and Mustafa Suleyman. Demis Hassabis is the co founder and chief executive officer of DeepMind. He was a child chess prodigy, then a video game designer, then a neuroscientist, before turning to artificial intelligence, with the stated mission to solve intelligence and use it to solve everything else.\n\n[pause for questions]\n\nCircling back to an earlier point, Welcome everyone, and thanks for joining this deep dive on artificial intelligence, its history, its present, and where it might be heading. We have a lot of ground to cover today, so let us get started right away. This is a long session, spanning many hours, so feel free to pause and come back at any point.\n\n[slide transition]\n\n[brief technical difficulty, resuming]\n\nCircling back to an earlier point, DeepMind is a Brit

In [39]:
parser = StrOutputParser()

In [40]:
# parallel_chain -> {context, question}  ->  prompt fills template  ->  llm answers  ->  parser -> string
main_chain = parallel_chain | prompt | llm | parser

In [41]:
# parallel_chain -> {context, question}  ->  prompt fills template  ->  llm answers  ->  parser -> string
main_chain = parallel_chain | prompt | llm | parser

In [42]:
main_chain.invoke('Can you summarize the video')
# question -> retrieve+format context -> fill prompt -> LLM answers grounded in transcript -> clean string

'The video discusses several key ideas:\n\n1. **Positional information in transformers** – because attention processes all tokens simultaneously, models need positional encodings (fixed sinusoidal or learned embeddings) to know each token’s place in the sequence.\n\n2. **Scope disclaimer** – it explicitly states that nuclear fusion is outside the session’s focus, which is on AI, DeepMind, and retrieval‑augmented generation (RAG).\n\n3. **Evaluating RAG systems** – evaluation is split into retrieval quality (getting the right chunks) and answer quality (accuracy, completeness, and faithfulness to the retrieved sources). Faithfulness is crucial since answers can sound convincing yet contradict their sources.\n\n4. **DeepMind’s achievements** – highlights AlphaGo’s breakthrough in defeating world‑champion Lee\u202fSedol in 2016, emphasizing the game’s vast combinatorial complexity.\n\n5. **Transformer attention advantage** – contrasts transformers with earlier sequential models, noting th